In [11]:
import geopandas as gpd
import numpy as np
import sys

from overturemaps import geodataframe

import osmnx as ox
import networkx as nx

from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform, pdist
from scipy.sparse.csgraph import connected_components
from scipy.sparse import coo_matrix
import shapely

from scipy.sparse import csr_matrix

In [12]:
sys.path.append('../src/nlr_ghe_screen/api/')
from osm_api import (
    get_overpass_with_splitting
)

sys.path.append('../src/nlr_ghe_screen/clustering')
from clustering import (
    get_clusters,
    group_ghes,
    construct_euclidian_dist_matrix,
    grow_cluster,
)

sys.path.append('../src/nlr_ghe_screen/processing/')
from ms_cross_validation import (
    get_ms_building_data,
    get_overturemaps_data,
)
from geometry import (
    combine_geometries
)

In [3]:
file_path = "../src/exports/"
gdf = gpd.read_file(file_path + 'ghe_locations.geojson')
ms_buildings = gpd.read_file('ms_buildings.geojson')

gdf = gdf.to_crs(gdf.estimate_utm_crs())
ms_buildings = ms_buildings.to_crs(gdf.crs)

In [5]:
dist_matrix = construct_euclidian_dist_matrix(
    gdf,
    max_distance=750
)

dense = dist_matrix.toarray()
dense[dense == 0] = 1e6
np.fill_diagonal(dense, 0)
labels = group_ghes(
    dist_matrix=dense,
    is_dense=True
)
gdf['seed_cluster'] = labels

Constructing distance matrix - euclidian


In [9]:
layers = []

for idx in range(300, 301):
    # idx = 200
    ghe_group = gdf[gdf['seed_cluster'] == idx].copy()
    group_union = ghe_group.geometry.union_all()
    group_centroid = ghe_group.geometry.union_all().centroid

    geoms = ms_buildings.geometry.to_numpy()
    buildings_tree = shapely.STRtree(geoms)

    area = shapely.box(*group_union.bounds).buffer(250)
    centroid_buffer = group_centroid.buffer(500)
    idx_nearby = buildings_tree.query(area, predicate='intersects')
    # idx_nearby = buildings_tree.query(centroid_buffer, predicate='dwithin', distance=100)

    layers.append(ghe_group)
    layers.append(ms_buildings.iloc[idx_nearby])

In [10]:
import itertools

colors = ['blue', 'red', 'green', 'orange', 'purple', 'black']

m = None
for i, (frame, color) in enumerate(zip(layers, itertools.cycle(colors))):
    m = frame.explore(m=m, color=color, name=f"layer_{i}")

m